# Deterministic seminar scenario videos

Run All executes the repository generator, displays the four-row metric table, previews the synchronized comparison, and provides links to every artifact.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import FileLink, Image, Video, display

def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'generate_seminar_videos.py').is_file() and (candidate / 'params').is_dir():
            return candidate
    raise RuntimeError('Could not locate the inverted_drone_sim repository root.')

ROOT = find_repository_root(Path.cwd())
OUTPUT_DIR = ROOT / 'results' / 'analysis' / 'seminar_videos'
PARAMETER_FILES = [
    ROOT / 'params' / 'loiter_tuned_vane_only.json',
    ROOT / 'params' / 'moving_mass_prototype_2kg_tuned.json',
]
ASSIST_GAIN_M_PER_NM = 0.055

display(pd.DataFrame({
    'selected_parameter_file': [str(path.relative_to(ROOT)) for path in PARAMETER_FILES],
    'exists': [path.is_file() for path in PARAMETER_FILES],
    'assist_gain_m_per_Nm': [0.0, ASSIST_GAIN_M_PER_NM],
}))

In [ ]:
completed = subprocess.run(
    [sys.executable, str(ROOT / 'generate_seminar_videos.py')],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)

In [ ]:
metrics = pd.read_csv(OUTPUT_DIR / 'scenario_metrics.csv')
display(metrics)

In [ ]:
composite_mp4 = OUTPUT_DIR / 'seminar_scenario_comparison.mp4'
composite_gif = OUTPUT_DIR / 'seminar_scenario_comparison.gif'
if composite_mp4.is_file():
    display(Video(str(composite_mp4), embed=False, html_attributes='controls width=960'))
else:
    print('MP4 unavailable; see manifest.json for the encoder warning.')
if composite_gif.is_file():
    display(Image(filename=str(composite_gif), width=960))

In [ ]:
artifact_names = [
    'seminar_scenario_comparison.mp4',
    'seminar_scenario_comparison.gif',
    'loiter_locked.mp4',
    'loiter_assist.mp4',
    'forward_1m_locked.mp4',
    'forward_1m_assist.mp4',
    'seminar_video_thumbnail.png',
    'scenario_metrics.csv',
    'scenario_summary.md',
    'manifest.json',
]
for name in artifact_names:
    path = OUTPUT_DIR / name
    if path.is_file():
        display(FileLink(str(path), result_html_prefix=f'{name}: '))
    else:
        print(f'{name}: not created')